In [ ]:
#IMPORT THƯ VIỆN
 
import os
import json
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns
 
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, WeightedRandomSampler
from torchvision import datasets, transforms, models
 
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    f1_score,
)
from sklearn.utils.class_weight import compute_class_weight
from pathlib import Path
 
print("Import hoàn tất.")
print(f"PyTorch version : {torch.__version__}")
print(f"CUDA available  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU             : {torch.cuda.get_device_name(0)}")

In [ ]:
# CẤU HÌNH
# Tập trung toàn bộ siêu tham số vào một chỗ để dễ điều chỉnh.
# Gồm 3 nhóm: đường dẫn, tham số train, tham số kỹ thuật.
 
SPLIT_PATH = "/kaggle/input/datasets/dunnguynvn/blood-data-l2/dataset_split"
OUTPUT_DIR = "/kaggle/working/resnet50_output"
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
 
CLASSES = [
    "BA", "BNE", "EO", "ERB", "IG", "LY", "MMY",
    "MO", "MY", "MYELOBLAST", "PLATELET", "PMY", "SNE"
]
NUM_CLASSES = len(CLASSES)
 
IMG_SIZE    = 224
BATCH_SIZE  = 32
NUM_WORKERS = 2
PIN_MEMORY  = True
 
EPOCHS_WARMUP    = 3
EPOCHS_FINETUNE1 = 5
EPOCHS_FINETUNE2 = 22
TOTAL_EPOCHS     = EPOCHS_WARMUP + EPOCHS_FINETUNE1 + EPOCHS_FINETUNE2
 
LR_HEAD      = 1e-3
LR_BACKBONE1 = 1e-4
LR_BACKBONE2 = 5e-5
 
WEIGHT_DECAY        = 1e-4
LABEL_SMOOTHING     = 0.1
FOCAL_GAMMA         = 2.0
EARLY_STOP_PATIENCE = 7
RANDOM_SEED         = 42
 
torch.manual_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
 
print(f"\nCấu hình:")
print(f"  Device       : {DEVICE}")
print(f"  Num classes  : {NUM_CLASSES}")
print(f"  Total epochs : {TOTAL_EPOCHS} "
      f"(warmup={EPOCHS_WARMUP}, ft1={EPOCHS_FINETUNE1}, ft2={EPOCHS_FINETUNE2})")
print(f"  Batch size   : {BATCH_SIZE}")
print(f"  Output dir   : {OUTPUT_DIR}")

In [ ]:
# DATA TRANSFORMS
# Train: thêm augmentation online nhẹ (lật, xoay, color jitter) để tăng
#        đa dạng dữ liệu trong mỗi epoch, sau đó chuẩn hóa theo ImageNet.
# Val/Test: chỉ resize + normalize, không augment để đánh giá khách quan
 
transform_train = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.RandomRotation(degrees=15),
    transforms.ColorJitter(brightness=0.1, contrast=0.1,
                           saturation=0.1, hue=0.05),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])
 
transform_val = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

In [ ]:
# DATASET & DATALOADER
# ImageFolder tự đọc nhãn từ tên thư mục con.
# WeightedRandomSampler giúp các lớp thiểu số (IG, PMY) xuất hiện
# nhiều hơn trong mỗi batch, bù đắp mất cân bằng dữ liệu.
# class_weights_tensor dùng trong loss để phạt nặng hơn khi sai lớp ít.
 
train_dataset = datasets.ImageFolder(
    root=os.path.join(SPLIT_PATH, "train"), transform=transform_train)
val_dataset = datasets.ImageFolder(
    root=os.path.join(SPLIT_PATH, "val"),   transform=transform_val)
test_dataset = datasets.ImageFolder(
    root=os.path.join(SPLIT_PATH, "test"),  transform=transform_val)
 
labels_array = np.array(train_dataset.targets)
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(labels_array),
    y=labels_array
)
class_weights_tensor = torch.FloatTensor(class_weights).to(DEVICE)
 
sample_weights = torch.DoubleTensor(
    [class_weights[t] for t in train_dataset.targets]
)
sampler = WeightedRandomSampler(
    weights=sample_weights, num_samples=len(sample_weights), replacement=True
)
 
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE,
                          sampler=sampler, num_workers=NUM_WORKERS,
                          pin_memory=PIN_MEMORY)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE,
                          shuffle=False,  num_workers=NUM_WORKERS,
                          pin_memory=PIN_MEMORY)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE,
                          shuffle=False,  num_workers=NUM_WORKERS,
                          pin_memory=PIN_MEMORY)
 
print(f"Dataset đã tải:")
print(f"  Train : {len(train_dataset)} ảnh")
print(f"  Val   : {len(val_dataset)} ảnh")
print(f"  Test  : {len(test_dataset)} ảnh")

In [ ]:
# FOCAL LOSS
# Cải tiến của CrossEntropyLoss cho bài toán mất cân bằng.
# Thêm hệ số (1-p)^gamma: mẫu dễ (p cao) → loss nhỏ, mẫu khó (p thấp)
# → loss lớn → model tập trung học các mẫu khó phân loại hơn.
# Kết hợp class_weight và label_smoothing để tăng hiệu quả.
 
class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0, weight=None, label_smoothing=0.0):
        super().__init__()
        self.gamma           = gamma
        self.weight          = weight
        self.label_smoothing = label_smoothing
 
    def forward(self, inputs, targets):
        ce_loss = nn.CrossEntropyLoss(
            weight=self.weight,
            label_smoothing=self.label_smoothing,
            reduction='none'
        )(inputs, targets)
        pt    = torch.exp(-ce_loss)
        focal = (1 - pt) ** self.gamma * ce_loss
        return focal.mean()

In [ ]:
# MIXUP AUGMENTATION
# Trộn 2 ảnh và nhãn theo tỉ lệ lambda ~ Beta(alpha, alpha).
# x_mixed = λ*x_i + (1-λ)*x_j  |  loss = λ*L(y_i) + (1-λ)*L(y_j)
# Buộc model học ranh giới quyết định mượt hơn, giảm overconfident,
# cải thiện generalization đặc biệt hiệu quả với lớp hình thái tương tự.
# Không dùng ở Warm-up vì head chưa ổn định.
 
def mixup_data(x, y, alpha=0.4):
    lam   = np.random.beta(alpha, alpha) if alpha > 0 else 1.0
    index = torch.randperm(x.size(0)).to(x.device)
    mixed_x = lam * x + (1 - lam) * x[index]
    return mixed_x, y, y[index], lam
 
def mixup_criterion(criterion, pred, y_a, y_b, lam):
    return lam * criterion(pred, y_a) + (1 - lam) * criterion(pred, y_b)

In [ ]:
# KIẾN TRÚC RESNET50
# Dùng ResNet50 pretrained ImageNet (IMAGENET1K_V2 tốt hơn V1 ~3%).
# Thay classifier head gốc (fc: 2048→1000) bằng head mới (2048→512→13).
# Dropout 0.3/0.2 để chống overfitting mà không gây underfitting.
# Không có Softmax vì FocalLoss đã tích hợp LogSoftmax bên trong.
 
def build_resnet50(num_classes):
    model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
    in_features = model.fc.in_features
    model.fc = nn.Sequential(
        nn.Dropout(p=0.3),
        nn.Linear(in_features, 512),
        nn.ReLU(inplace=True),
        nn.Dropout(p=0.2),
        nn.Linear(512, num_classes)
    )
    return model
 
model = build_resnet50(NUM_CLASSES).to(DEVICE)
 
total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nMô hình ResNet50:")
print(f"  Tổng tham số      : {total_params:,}")
print(f"  Tham số trainable : {trainable_params:,}")

In [ ]:
# HÀM FREEZE / UNFREEZE
# Progressive unfreezing mở dần từng layer theo giai đoạn:
#   Warmup   : freeze toàn bộ backbone, chỉ train head
#              → Tránh phá pretrained weights khi head chưa ổn định
#   Finetune1: mở layer4 (đặc trưng cao cấp: shape, cấu trúc tế bào)
#   Finetune2: mở thêm layer3 (đặc trưng mid-level: texture, edge)
#              → Quan trọng để phân biệt tế bào hình thái tương tự
 
def freeze_backbone(model):
    for name, param in model.named_parameters():
        if "fc" not in name:
            param.requires_grad = False
 
def unfreeze_layer4(model):
    for name, param in model.named_parameters():
        if "layer4" in name or "fc" in name:
            param.requires_grad = True
 
def unfreeze_layer3_layer4(model):
    for name, param in model.named_parameters():
        if "layer3" in name or "layer4" in name or "fc" in name:
            param.requires_grad = True
 
def count_trainable(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

In [ ]:
# HÀM TRAIN VÀ VALIDATE MỘT EPOCH
# train_one_epoch: model.train() → bật Dropout, cập nhật gradient.
#                  Hỗ trợ MixUp để trộn ảnh/nhãn trong batch.
# validate       : model.eval() + torch.no_grad() → không cập nhật gradient.
#                  Tính loss, accuracy, F1 macro và thu thập dự đoán
#                  để vẽ confusion matrix sau này.
 
criterion = FocalLoss(
    gamma=FOCAL_GAMMA,
    weight=class_weights_tensor,
    label_smoothing=LABEL_SMOOTHING
)
 
def train_one_epoch(model, loader, optimizer, use_mixup=True):
    model.train()
    total_loss = total_correct = total_samples = 0
 
    for images, labels in loader:
        images = images.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)
 
        if use_mixup:
            images, labels_a, labels_b, lam = mixup_data(images, labels)
            outputs = model(images)
            loss    = mixup_criterion(criterion, outputs, labels_a, labels_b, lam)
        else:
            outputs = model(images)
            loss    = criterion(outputs, labels)
 
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
 
        total_loss    += loss.item() * images.size(0)
        preds          = outputs.argmax(dim=1)
        total_correct += (preds == labels).sum().item()
        total_samples += images.size(0)
 
    return total_loss / total_samples, total_correct / total_samples
 
 
def validate(model, loader):
    model.eval()
    total_loss = total_correct = total_samples = 0
    all_preds  = []
    all_labels = []
 
    with torch.no_grad():
        for images, labels in loader:
            images = images.to(DEVICE, non_blocking=True)
            labels = labels.to(DEVICE, non_blocking=True)
 
            outputs = model(images)
            loss    = criterion(outputs, labels)
            preds   = outputs.argmax(dim=1)
 
            total_loss    += loss.item() * images.size(0)
            total_correct += (preds == labels).sum().item()
            total_samples += images.size(0)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
 
    avg_loss = total_loss / total_samples
    avg_acc  = total_correct / total_samples
    f1_macro = f1_score(all_labels, all_preds, average='macro', zero_division=0)
    return avg_loss, avg_acc, f1_macro, all_preds, all_labels

In [ ]:
# VÒNG LẶP TRAIN CHÍNH
# Chạy qua 3 giai đoạn, mỗi giai đoạn có freeze/unfreeze và LR riêng.
# Mỗi epoch ghi đầy đủ: loss, acc, f1, lr, thời gian vào history.
# Checkpoint lưu model tốt nhất theo val_acc.
# Early stopping dừng nếu val_acc không cải thiện sau N epoch liên tiếp.
 
history = {
    "epoch": [], "stage": [], "train_loss": [], "train_acc": [],
    "val_loss": [], "val_acc": [], "val_f1": [], "lr": [], "epoch_time": [],
}
 
best_val_acc    = 0.0
early_stop_cnt  = 0
best_model_path = os.path.join(OUTPUT_DIR, "resnet50_best.pth")
 
print("\n" + "=" * 70)
print("BẮT ĐẦU TRAIN RESNET50")
print("=" * 70)
 
global_epoch = 0
 
for stage_name, n_epochs in [
    ("Warmup",    EPOCHS_WARMUP),
    ("Finetune1", EPOCHS_FINETUNE1),
    ("Finetune2", EPOCHS_FINETUNE2),
]:
    if stage_name == "Warmup":
        freeze_backbone(model)
        optimizer = optim.AdamW(
            filter(lambda p: p.requires_grad, model.parameters()),
            lr=LR_HEAD, weight_decay=WEIGHT_DECAY
        )
        print(f"\n[Giai đoạn 1: Warm-up] Freeze backbone | "
              f"LR head={LR_HEAD} | Epochs={n_epochs}")
 
    elif stage_name == "Finetune1":
        unfreeze_layer4(model)
        optimizer = optim.AdamW([
            {"params": model.fc.parameters(), "lr": LR_HEAD},
            {"params": [p for n, p in model.named_parameters()
                        if "layer4" in n and p.requires_grad],
             "lr": LR_BACKBONE1},
        ], weight_decay=WEIGHT_DECAY)
        print(f"\n[Giai đoạn 2: Fine-tune layer4] "
              f"LR backbone={LR_BACKBONE1} | LR head={LR_HEAD} | Epochs={n_epochs}")
 
    else:
        unfreeze_layer3_layer4(model)
        optimizer = optim.AdamW([
            {"params": model.fc.parameters(), "lr": LR_HEAD / 2},
            {"params": [p for n, p in model.named_parameters()
                        if ("layer3" in n or "layer4" in n) and p.requires_grad],
             "lr": LR_BACKBONE2},
        ], weight_decay=WEIGHT_DECAY)
        print(f"\n[Giai đoạn 3: Fine-tune layer3+4] "
              f"LR backbone={LR_BACKBONE2} | LR head={LR_HEAD/2} | Epochs={n_epochs}")
 
    print(f"  Tham số trainable: {count_trainable(model):,}")
 
    scheduler = optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=n_epochs, eta_min=1e-6
    )
    use_mixup = (stage_name != "Warmup")
 
    for ep in range(n_epochs):
        global_epoch += 1
        t0 = time.time()
 
        train_loss, train_acc = train_one_epoch(
            model, train_loader, optimizer, use_mixup=use_mixup)
        val_loss, val_acc, val_f1, _, _ = validate(model, val_loader)
        scheduler.step()
 
        elapsed = time.time() - t0
        cur_lr  = optimizer.param_groups[0]["lr"]
 
        history["epoch"].append(global_epoch)
        history["stage"].append(stage_name)
        history["train_loss"].append(round(train_loss, 4))
        history["train_acc"].append(round(train_acc, 4))
        history["val_loss"].append(round(val_loss, 4))
        history["val_acc"].append(round(val_acc, 4))
        history["val_f1"].append(round(val_f1, 4))
        history["lr"].append(round(cur_lr, 7))
        history["epoch_time"].append(round(elapsed, 1))
 
        print(f"  Epoch [{global_epoch:02d}/{TOTAL_EPOCHS}] ({stage_name}) | "
              f"Train loss={train_loss:.4f} acc={train_acc:.4f} | "
              f"Val loss={val_loss:.4f} acc={val_acc:.4f} F1={val_f1:.4f} | "
              f"LR={cur_lr:.2e} | {elapsed:.1f}s")
 
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save({
                "epoch"      : global_epoch,
                "stage"      : stage_name,
                "model_state": model.state_dict(),
                "optimizer"  : optimizer.state_dict(),
                "val_acc"    : val_acc,
                "val_f1"     : val_f1,
            }, best_model_path)
            print(f"   Lưu best model (val_acc={val_acc:.4f})")
            early_stop_cnt = 0
        else:
            early_stop_cnt += 1
            if early_stop_cnt >= EARLY_STOP_PATIENCE:
                print(f"\n   Early stopping tại epoch {global_epoch} "
                      f"(không cải thiện {EARLY_STOP_PATIENCE} epochs liên tiếp)")
                break
 
print("\n" + "=" * 70)
print(f"TRAIN HOÀN TẤT | Best val accuracy: {best_val_acc:.4f}")
print("=" * 70)

In [ ]:
# ĐÁNH GIÁ TRÊN TẬP TEST
# Load best model 
checkpoint = torch.load(best_model_path, map_location=DEVICE)
model.load_state_dict(checkpoint["model_state"])
print(f"\nLoad best model từ epoch {checkpoint['epoch']} "
      f"(val_acc={checkpoint['val_acc']:.4f})")
 
test_loss, test_acc, test_f1, test_preds, test_labels = validate(
    model, test_loader)
 
print(f"\nKết quả trên tập TEST:")
print(f"  Loss     : {test_loss:.4f}")
print(f"  Accuracy : {test_acc:.4f}  ({test_acc*100:.2f}%)")
print(f"  F1 Macro : {test_f1:.4f}")
 
report = classification_report(
    test_labels, test_preds, target_names=CLASSES, digits=4)
print(f"\nClassification Report:\n{report}")
 

In [ ]:
# XUẤT SỐ LIỆU
# Xuất 3 file:
#   training_history.csv      : bảng loss/acc/f1 theo từng epoch
#   classification_report.csv : precision/recall/f1 theo từng lớp tế bào
#   summary.json              : tóm tắt kết quả + toàn bộ siêu tham số

df_history = pd.DataFrame(history)
df_history.to_csv(os.path.join(OUTPUT_DIR, "training_history.csv"), index=False)
print(f"Đã lưu: training_history.csv")
 
report_dict = classification_report(
    test_labels, test_preds, target_names=CLASSES, digits=4, output_dict=True)
df_report = pd.DataFrame(report_dict).transpose().round(4)
df_report.to_csv(os.path.join(OUTPUT_DIR, "classification_report.csv"))
print(f"Đã lưu: classification_report.csv")
 
summary = {
    "model"            : "ResNet50",
    "best_epoch"       : int(checkpoint["epoch"]),
    "best_val_acc"     : round(float(checkpoint["val_acc"]), 4),
    "best_val_f1"      : round(float(checkpoint["val_f1"]), 4),
    "test_loss"        : round(test_loss, 4),
    "test_accuracy"    : round(test_acc, 4),
    "test_accuracy_pct": round(test_acc * 100, 2),
    "test_f1_macro"    : round(test_f1, 4),
    "total_epochs_run" : global_epoch,
    "config": {
        "img_size"         : IMG_SIZE,
        "batch_size"       : BATCH_SIZE,
        "focal_gamma"      : FOCAL_GAMMA,
        "label_smoothing"  : LABEL_SMOOTHING,
        "weight_decay"     : WEIGHT_DECAY,
        "epochs_warmup"    : EPOCHS_WARMUP,
        "epochs_finetune1" : EPOCHS_FINETUNE1,
        "epochs_finetune2" : EPOCHS_FINETUNE2,
        "lr_head"          : LR_HEAD,
        "lr_backbone1"     : LR_BACKBONE1,
        "lr_backbone2"     : LR_BACKBONE2,
    }
}
with open(os.path.join(OUTPUT_DIR, "summary.json"), "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2, ensure_ascii=False)
print(f"Đã lưu: summary.json")
 
print("\n" + "=" * 70)
print("BẢNG KẾT QUẢ PER-CLASS (copy vào luận văn)")
print("=" * 70)
print(f"  {'Lớp':<15} {'Precision':>10} {'Recall':>10} {'F1-score':>10} {'Support':>10}")
print(f"  {'-'*57}")
for cls in CLASSES:
    p  = report_dict[cls]["precision"]
    r  = report_dict[cls]["recall"]
    f1 = report_dict[cls]["f1-score"]
    s  = int(report_dict[cls]["support"])
    print(f"  {cls:<15} {p:>10.4f} {r:>10.4f} {f1:>10.4f} {s:>10}")
print(f"  {'-'*57}")
print(f"  {'Accuracy':<15} {'':>10} {'':>10} {test_acc:>10.4f} {len(test_labels):>10}")
print(f"  {'Macro avg':<15} "
      f"{report_dict['macro avg']['precision']:>10.4f} "
      f"{report_dict['macro avg']['recall']:>10.4f} "
      f"{report_dict['macro avg']['f1-score']:>10.4f} "
      f"{len(test_labels):>10}")
print(f"  {'Weighted avg':<15} "
      f"{report_dict['weighted avg']['precision']:>10.4f} "
      f"{report_dict['weighted avg']['recall']:>10.4f} "
      f"{report_dict['weighted avg']['f1-score']:>10.4f} "
      f"{len(test_labels):>10}")

In [ ]:
# VẼ BIỂU ĐỒ
# Xuất 2 biểu đồ:
#   training_curves.png  
#   confusion_matrix.png 
 
stage_colors = {"Warmup": "#FFF3CD", "Finetune1": "#D1ECF1", "Finetune2": "#D4EDDA"}
 
def shade_stages(ax):
    prev, start = None, 1
    for ep, st in zip(history["epoch"], history["stage"]):
        if st != prev:
            if prev:
                ax.axvspan(start-0.5, ep-0.5, alpha=0.15,
                           color=stage_colors[prev], label=f"Stage: {prev}")
            start, prev = ep, st
    ax.axvspan(start-0.5, history["epoch"][-1]+0.5, alpha=0.15,
               color=stage_colors[prev], label=f"Stage: {prev}")
 
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
epochs_x = history["epoch"]
 
ax = axes[0]
shade_stages(ax)
ax.plot(epochs_x, history["train_loss"], label="Train Loss", color="#E74C3C", linewidth=2)
ax.plot(epochs_x, history["val_loss"],   label="Val Loss",   color="#3498DB", linewidth=2)
ax.set_title("Loss theo Epoch", fontsize=13, fontweight='bold')
ax.set_xlabel("Epoch"); ax.set_ylabel("Loss")
ax.legend(); ax.grid(True, alpha=0.3)
 
ax = axes[1]
shade_stages(ax)
ax.plot(epochs_x, [v*100 for v in history["train_acc"]], label="Train Acc", color="#E74C3C", linewidth=2)
ax.plot(epochs_x, [v*100 for v in history["val_acc"]],   label="Val Acc",   color="#3498DB", linewidth=2)
ax.set_title("Accuracy theo Epoch", fontsize=13, fontweight='bold')
ax.set_xlabel("Epoch"); ax.set_ylabel("Accuracy (%)")
ax.yaxis.set_major_formatter(ticker.FormatStrFormatter('%.1f'))
ax.legend(); ax.grid(True, alpha=0.3)
 
ax = axes[2]
shade_stages(ax)
ax.plot(epochs_x, history["val_f1"], label="Val F1 Macro", color="#27AE60", linewidth=2)
ax.set_title("F1 Macro theo Epoch", fontsize=13, fontweight='bold')
ax.set_xlabel("Epoch"); ax.set_ylabel("F1 Score")
ax.legend(); ax.grid(True, alpha=0.3)
 
plt.suptitle("ResNet50 — Training Curves", fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "training_curves.png"), dpi=150, bbox_inches='tight')
plt.show()
print("Đã lưu: training_curves.png")
 
cm      = confusion_matrix(test_labels, test_preds)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
 
fig, axes = plt.subplots(1, 2, figsize=(22, 9))
for ax, data, fmt, title in zip(
    axes,
    [cm, cm_norm], ["d", ".2f"],
    ["Confusion Matrix (Số lượng)", "Confusion Matrix (Tỉ lệ)"]
):
    sns.heatmap(data, annot=True, fmt=fmt,
                xticklabels=CLASSES, yticklabels=CLASSES,
                cmap="Blues", ax=ax, linewidths=0.5, annot_kws={"size": 8})
    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.set_xlabel("Predicted", fontsize=11)
    ax.set_ylabel("Actual", fontsize=11)
    ax.tick_params(axis='x', rotation=45)
    ax.tick_params(axis='y', rotation=0)
 
plt.suptitle("ResNet50 — Confusion Matrix trên tập Test",
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "confusion_matrix.png"), dpi=150, bbox_inches='tight')
plt.show()
print("Đã lưu: confusion_matrix.png")

In [ ]:
# TỔNG KẾT OUTPUT
# Kiểm tra tất cả file đã được xuất thành công.
# File resnet50_best.pth sẽ được dùng trong notebook Ensemble.
 
print("\n" + "=" * 70)
print("TẤT CẢ FILE OUTPUT")
print("=" * 70)
for fname, desc in {
    "resnet50_best.pth"         : "Best model weights → dùng cho Ensemble",
    "training_history.csv"      : "Loss/Acc/F1 từng epoch → bảng báo cáo",
    "classification_report.csv" : "Precision/Recall/F1 từng lớp → bảng báo cáo",
    "summary.json"              : "Tóm tắt kết quả + cấu hình",
    "training_curves.png"       : "Biểu đồ Loss/Acc/F1 → hình báo cáo",
    "confusion_matrix.png"      : "Ma trận nhầm lẫn → hình báo cáo",
}.items():
    fpath  = os.path.join(OUTPUT_DIR, fname)
    exists = "ok" if os.path.exists(fpath) else "ko ok"
    print(f"  {exists} {fname:<35} → {desc}")
 
print(f"\nKết quả cuối cùng ResNet50:")
print(f"  Test Accuracy : {test_acc*100:.2f}%")
print(f"  Test F1 Macro : {test_f1:.4f}")
print(f"  Best epoch    : {checkpoint['epoch']}")